In [1]:
import pandas as pd
import numpy as np
import os

# 1. Pfade definieren
raw_file_path = '../data/raw/olympics.xlsx'
processed_dir = '../data/processed/'

os.makedirs(processed_dir, exist_ok=True)

# 2. Rohdaten laden
print("Lade Excel-Rohdaten...")
# Falls die Excel-Datei mehrere Tabellenblätter enthält, ggf. sheet_name anpassen
df_raw = pd.read_excel(raw_file_path)

# 3. Grundlegende Datenbereinigung
print("Starte Datenbereinigung...")
df = df_raw.copy()

# Spaltennamen bereinigen (Leerzeichen entfernen, Kleinbuchstaben)
df.columns = df.columns.str.strip()

# Duplikate entfernen
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplikate entfernt: {initial_rows - len(df)} Zeilen gelöscht.")

# Fehlende Werte (NaN) bei Medaillen auffüllen
if 'Medal' in df.columns:
    df['Medal'] = df['Medal'].fillna('No Medal')

# Numerische Spalten bereinigen (Alter, Größe, Gewicht)
numeric_cols = ['Age', 'Height', 'Weight', 'Year']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Textspalten bereinigen
string_cols = ['Name', 'Sex', 'Team', 'NOC', 'Games', 'Season', 'City', 'Sport', 'Event']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# 4. Erstellung der Hauptdatei (olympics_cleaned.csv)
cleaned_file_path = os.path.join(processed_dir, 'olympics_cleaned.csv')
df.to_csv(cleaned_file_path, index=False)
print(f"Bereinigte Gesamtdaten gespeichert unter: {cleaned_file_path}")

# 5. Aufteilung in relationale Tabellen (Athletes, Events, NOC Regions)
print("Erstelle relationale Teil-Datensätze...")

# A) Athletes Table
if set(['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight']).issubset(df.columns):
    df_athletes = df[['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight']].drop_duplicates(subset=['ID'])
    df_athletes.to_csv(os.path.join(processed_dir, 'athletes.csv'), index=False)
    print("-> athletes.csv erstellt.")

# B) Events Table
if set(['Event', 'Sport']).issubset(df.columns):
    df_events = df[['Event', 'Sport']].drop_duplicates()
    df_events['event_id'] = range(1, len(df_events) + 1)
    df_events.to_csv(os.path.join(processed_dir, 'events.csv'), index=False)
    print("-> events.csv erstellt.")

# C) NOC Regions Table
if set(['NOC', 'Team']).issubset(df.columns):
    df_noc = df[['NOC', 'Team']].drop_duplicates(subset=['NOC']).rename(columns={'Team': 'region_name'})
    df_noc.to_csv(os.path.join(processed_dir, 'noc_regions.csv'), index=False)
    print("-> noc_regions.csv erstellt.")

print("\nBereinigung erfolgreich abgeschlossen!")

Lade Excel-Rohdaten...
Starte Datenbereinigung...
Duplikate entfernt: 0 Zeilen gelöscht.
Bereinigte Gesamtdaten gespeichert unter: ../data/processed/olympics_cleaned.csv
Erstelle relationale Teil-Datensätze...
-> athletes.csv erstellt.
-> events.csv erstellt.

Bereinigung erfolgreich abgeschlossen!
